# Local Model Evaluation
This notebook downloads the original datasets locally, prepares them, and evaluates the `best.pt` model.

**Note:** You will need your Roboflow API key in the next cell.

In [1]:
# Install roboflow locally if you haven't already
!uv pip install roboflow pyyaml


Using Python 3.12.11 environment at: /Users/bahacelik/Documents/Thesis/solar-sentinel/src/.venv
Audited 2 packages in 58ms


In [2]:
import os
from pathlib import Path

# ============================================================
# CONFIGURATION — Set your Roboflow API key here
# Get a free key at: https://app.roboflow.com/settings/api
# ============================================================
ROBOFLOW_API_KEY = "spTebOB7Bo1oCViX9nws"  # <-- Paste your API key here

# Base directory for all datasets
BASE_DIR = Path("datasets/raw")
BASE_DIR.mkdir(exist_ok=True)

# Dataset definitions: (workspace, project, version)
# These are the 3 best RGB solar panel defect datasets on Roboflow
DATASETS = [
    {
        "name": "dataset_1",
        "workspace": "solar-panel-defect-detection",
        "project": "solar-panel-defect-detection-hpser",
        "version": 1,
    },
    {
        "name": "dataset_2",
        "workspace": "solar-panel-defect-ufk3b",
        "project": "solar-panel-defect-2-aeuek",
        "version": 4,
    },
    {
        "name": "dataset_3",
        "workspace": "gao-shou-zheng-b6xqc",
        "project": "solar-panel-0swal",
        "version": 3,
    },
]

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    for ds in DATASETS:
        print(f"\n📥 Downloading {ds['name']}...")
        project = rf.workspace(ds["workspace"]).project(ds["project"])
        dataset = project.version(ds["version"]).download(
            "yolov8",
            location=str(BASE_DIR / ds["name"]),
        )
        print(f"   ✅ Saved to {BASE_DIR / ds['name']}")

    print("\n✅ All datasets downloaded!")
else:
    print("⚠️  No Roboflow API key set.")
    print("   Option 1: Set ROBOFLOW_API_KEY above and re-run")
    print("   Option 2: Manually download from Roboflow Universe:")
    print("     • https://universe.roboflow.com/solar-panel-defect-detection/solar-panel-defect-detection-hpser")
    print("     • https://universe.roboflow.com/internship-project-rvpro/solar-panel-defect-btmyj")
    print("     • https://universe.roboflow.com/solar-panel-defects-ax0gw/solar-panel-defect-w1dfi")
    print("   Upload zips to /content/datasets/ and extract them.")



📥 Downloading dataset_1...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to datasets/raw/dataset_1 in yolov8:: 100%|██████████| 3390/3390 [00:00<00:00, 5818.80it/s]


   ✅ Saved to datasets/raw/dataset_1

📥 Downloading dataset_2...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to datasets/raw/dataset_2 in yolov8:: 100%|██████████| 2389/2389 [00:00<00:00, 6974.81it/s]

   ✅ Saved to datasets/raw/dataset_2

📥 Downloading dataset_3...
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to datasets/raw/dataset_3 in yolov8:: 100%|██████████| 4572/4572 [00:00<00:00, 8208.75it/s]

   ✅ Saved to datasets/raw/dataset_3

✅ All datasets downloaded!


In [ ]:
import shutil
import yaml
import hashlib
from pathlib import Path
from collections import Counter

# ============================================================
# CLASS REMAPPING RULES
# Maps every possible original class name → our 3-class system
# ============================================================
CLASS_REMAP = {
    # --- DAMAGE (class 0) ---
    # Physical damage: cracks, broken glass, deformation
    "defective": 0,
    "physical-damage": 0,
    "physical damage": 0,
    "physical_damage": 0,
    "crack": 0,
    # Electrical damage: burnt cells, hotspots, wiring issues
    "electrical-damage": 0,
    "electrical damage": 0,
    "electrical_damage": 0,
    # Generic damage labels
    "damage": 0,
    "damage panel": 0,

    # --- BLOCKAGE → DEFECT (class 0) ---
    # Bird droppings
    "bird-drop": 0,
    "bird drop": 0,
    "bird-drop panel": 0,
    # Dust/dirt accumulation
    "dusty": 0,
    "dusty panel": 0,
    "dust": 0,
    "cover": 0,
    # Snow coverage
    "snow": 0,
    "snow-covered": 0,
    "snow-covered panel": 0,

    # --- HEALTHY (class 1) ---
    "non-defective": 1,
    "non defective": 1,
    "clean": 1,
    "clean panel": 1,
    "normal": 1,
}

# Our target class names (order matters — matches Class IDs above)
TARGET_CLASSES = ["defect", "healthy"]

# Output directory for the merged dataset
MERGED_DIR = Path("datasets/solar-sentinel")


def read_dataset_yaml(dataset_path: Path) -> dict:
    """Read the data.yaml from a Roboflow-exported dataset.

    This file contains the original class names and their IDs.
    Example content:
        names:
          0: Bird Drop
          1: Defective
          2: Dusty
    """
    yaml_path = dataset_path / "data.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError(f"No data.yaml found at {yaml_path}")

    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    # Roboflow exports class names as {id: name} dict
    return data.get("names", {})


def build_id_remap(original_names: dict) -> dict:
    """Build old_id → new_id mapping for a specific dataset.

    Args:
        original_names: {0: 'Bird Drop', 1: 'Defective', ...}

    Returns:
        {0: 1, 1: 0, 2: 1, ...}  (old_id → new_id)
    """
    remap = {}
    if isinstance(original_names, list):
        original_names = {i: name for i, name in enumerate(original_names)}
    for old_id, name in original_names.items():
        # Normalize: lowercase + strip whitespace for matching
        normalized = name.lower().strip()
        if normalized in CLASS_REMAP:
            remap[int(old_id)] = CLASS_REMAP[normalized]
        else:
            print(f"  ⚠️  Unknown class '{name}' (id={old_id}) — skipping")
    return remap


def remap_annotation_file(txt_path: Path, id_remap: dict) -> list:
    """Rewrite a single YOLO annotation file with remapped class IDs.

    YOLO format: <class_id> <x_center> <y_center> <width> <height>
    We only change the class_id, keeping bounding box coordinates intact.

    Returns list of new class IDs found (for counting).
    """
    lines = txt_path.read_text().strip().splitlines()
    new_lines = []
    class_ids_found = []

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue  # Skip malformed lines

        old_id = int(parts[0])
        if old_id not in id_remap:
            continue  # Skip unknown classes

        new_id = id_remap[old_id]
        # Replace old class ID with new one, keep bbox coordinates
        new_lines.append(f"{new_id} {' '.join(parts[1:])}")
        class_ids_found.append(new_id)

    txt_path.write_text("\n".join(new_lines) + "\n" if new_lines else "")
    return class_ids_found


def merge_datasets():
    """Merge all downloaded datasets into a single unified dataset."""
    # Create output structure
    for split in ["train", "val", "test"]:
        (MERGED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (MERGED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    total_stats = Counter()
    total_images = 0
    seen_hashes = set()  # For deduplication by content hash

    for ds in DATASETS:
        ds_path = BASE_DIR / ds["name"]

        # Find the actual dataset root (Roboflow may nest it)
        # Look for data.yaml to find the root
        yaml_candidates = list(ds_path.rglob("data.yaml"))
        if not yaml_candidates:
            print(f"⚠️  Skipping {ds['name']}: no data.yaml found")
            continue
        ds_root = yaml_candidates[0].parent

        print(f"\n📂 Processing {ds['name']} ({ds_root})")

        # Read original class mapping and build remap
        original_names = read_dataset_yaml(ds_root)
        print(f"   Original classes: {original_names}")
        id_remap = build_id_remap(original_names)
        print(f"   Remap: {id_remap}")

        # Process each split (train/val/test)
        for split in ["train", "valid", "test"]:
            # Roboflow uses 'valid' not 'val'
            out_split = "val" if split == "valid" else split

            img_dir = ds_root / split / "images"
            lbl_dir = ds_root / split / "labels"

            if not img_dir.exists():
                continue

            images = [f for f in img_dir.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
            for img_path in images:
                # Deduplicate identical images by their actual content hash
                file_hash = hashlib.md5(img_path.read_bytes()).hexdigest()
                if file_hash in seen_hashes:
                    continue
                seen_hashes.add(file_hash)

                # Prefix filename with dataset name to avoid collisions
                safe_name = f"{ds['name']}_{img_path.name}"
                lbl_name = f"{ds['name']}_{img_path.stem}.txt"

                # Copy image
                dst_img = MERGED_DIR / "images" / out_split / safe_name
                shutil.copy2(img_path, dst_img)

                # Copy and remap label
                src_lbl = lbl_dir / f"{img_path.stem}.txt"
                dst_lbl = MERGED_DIR / "labels" / out_split / lbl_name

                if src_lbl.exists():
                    shutil.copy2(src_lbl, dst_lbl)
                    ids = remap_annotation_file(dst_lbl, id_remap)
                    for cid in ids:
                        total_stats[TARGET_CLASSES[cid]] += 1
                else:
                    # No annotation = empty file (YOLO interprets as no objects)
                    dst_lbl.write_text("")

                total_images += 1

    # Print summary
    print(f"\n{'='*50}")
    print(f"✅ Merged dataset: {total_images} images")
    print(f"   📁 Location: {MERGED_DIR}")
    print(f"\n   Class distribution (annotations):")
    for cls_name, count in total_stats.most_common():
        print(f"      {cls_name}: {count}")

    # Count per split
    for split in ["train", "val", "test"]:
        n = len(list((MERGED_DIR / "images" / split).glob("*.*")))
        print(f"   {split}: {n} images")


# Run the merge!
merge_datasets()


In [ ]:
import yaml
from pathlib import Path

# Dataset configuration for YOLO26n training
# This tells ultralytics where the data is and what classes to detect
dataset_config = {
    "path": str(MERGED_DIR),          # Root directory of the dataset
    "train": "images/train",          # Training images (relative to path)
    "val": "images/val",              # Validation images
    "test": "images/test",            # Test images (optional)
    "nc": 2,                          # Number of classes
    "names": ["defect", "healthy"],  # Class names (order = class ID)
}

# Write the YAML file
yaml_path = MERGED_DIR / "solar_sentinel.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

# Display the generated config
print("📄 Generated dataset config:\n")
print(yaml_path.read_text())
print(f"\n✅ Saved to: {yaml_path}")

In [ ]:
from ultralytics import YOLO

# Load the best model from training
best_model = YOLO("best.pt")

# Run validation on the validation set
metrics = best_model.val(
    data=str(MERGED_DIR / "solar_sentinel.yaml"),
    imgsz=640,
    verbose=True,
)

# Print results in a readable format
print("\n" + "="*60)
print("📊 VALIDATION RESULTS")
print("="*60)
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")
print(f"  mAP@50:     {metrics.box.map50:.4f}")
print(f"  mAP@50-95:  {metrics.box.map:.4f}")
print("\n  Per-class mAP@50:")
for i, cls_name in enumerate(TARGET_CLASSES):
    if i < len(metrics.box.maps):
        print(f"    {cls_name}: {metrics.box.maps[i]:.4f}")
print("="*60)

# Quick assessment
if metrics.box.map50 > 0.5:
    print("\n✅ Model looks good! Proceeding to export.")
elif metrics.box.map50 > 0.3:
    print("\n⚠️  Model is decent but could improve. Consider:")
    print("   - Training for more epochs")
    print("   - Adding more training data")
    print("   - Adjusting augmentation settings")
else:
    print("\n❌ Model performance is low. Try:")
    print("   - Checking annotation quality (Cell 6)")
    print("   - Using a larger model (yolo26s.pt instead of nano)")
    print("   - Increasing augmentation")

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import random

model = YOLO("best.pt")

test_dir = MERGED_DIR / "images" / "test"
test_images = list(test_dir.glob("*.*"))
if not test_images:
    test_dir = MERGED_DIR / "images" / "val"
    test_images = list(test_dir.glob("*.*"))

samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    results = model.predict(source=str(img_path), imgsz=640, conf=0.25, verbose=False)
    annotated = results[0].plot()
    ax.imshow(annotated[:, :, ::-1])
    n_detections = len(results[0].boxes)
    class_names = [results[0].names[int(c)] for c in results[0].boxes.cls]
    title = f"{n_detections} detections: {', '.join(class_names) if class_names else 'none'}"
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.suptitle("🔍 Model Predictions on Test Images", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("inference_results.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Inference test complete. Check the images above.")
print("   If detections look correct → proceed to export!")